# M4 · Clustering-Pipeline für Housing-Daten

**Aufbau des Notebooks**

1. **Konfiguration** – alle zentralen Stellschrauben (inkl. Feature-Ausschluss)
2. **Setup & Daten laden**
3. **Kurze EDA** – Verteilungen & Korrelationen
4. **Helferfunktionen** – Kennzahlen, Cluster-Plots, Silhouette-Diagramme, Cohen's Kappa
5. **Algorithmen-Vergleich** – K-Means · Ward · GMM · DBSCAN · HDBSCAN (+ Ausreißer) · t-SNE/UMAP; je Verfahren wird die Clusterzahl **empirisch** bestimmt (Silhouette · Davies-Bouldin)
6. **Bewertung & Vergleich** – Kennzahlen, Silhouette-Diagramme aller Verfahren, Cohen's Kappa, Auswahl
7. **Cluster-Profile & externe Validierung** – inkl. der in §1 ausgeschlossenen Features

## 1 · Konfiguration
Alle zentralen Parameter an EINER Stelle. Besonders wichtig:

**`EXCLUDED_FEATURES`** – Features, die hier eingetragen werden, fließen **nicht** ins Clustering ein.
Sie werden am Ende (§7) als **zusätzliche externe Validierung** genutzt: Trennen sich die Cluster auch
auf Features, die der Algorithmus nie gesehen hat, sind die Segmente robuster als reine Geometrie.

In [ ]:
# ----------------------------- zentrale Konfiguration ------------------------------
RANDOM_STATE = 20
SIL_SAMPLE   = 3000      # Stichprobe für Silhouette (O(n^2) -> auf 21k zu teuer)
K_RANGE      = range(2, 9)  # Scan-Bereich für die empirische Bestimmung der Clusterzahl
MAX_NOISE_FRAC = 0.30    # Verfahren mit > 30 % Noise gelten nicht als vollständige Segmentierung

# Features, die vom Clustering AUSGESCHLOSSEN werden und in §7 zur Validierung dienen.
# Namen müssen exakt Spaltennamen des Datensatzes sein, z. B.:
#   EXCLUDED_FEATURES = ["sqft_lot", "floors"]
EXCLUDED_FEATURES = []

## 2 · Setup & Daten laden

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, HDBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             davies_bouldin_score, cohen_kappa_score,
                             adjusted_rand_score)
from sklearn.neighbors import NearestNeighbors

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

In [ ]:
df = pd.read_csv("cleaned_house_data_new.csv")

# ---- Spalten-Aufteilung: Clustering vs. Profil (prof_) vs. ausgeschlossene Features ----
prof_cols        = [c for c in df.columns if c.startswith("prof_")]
all_feature_cols = [c for c in df.columns if c not in prof_cols]

_unknown = sorted(set(EXCLUDED_FEATURES) - set(all_feature_cols))
assert not _unknown, f"EXCLUDED_FEATURES nicht im Datensatz gefunden: {_unknown}"

excluded_cols  = [c for c in all_feature_cols if c in EXCLUDED_FEATURES]
clustering_cols = [c for c in all_feature_cols if c not in EXCLUDED_FEATURES]
assert len(clustering_cols) >= 2, "Zu viele Features ausgeschlossen - mind. 2 müssen bleiben."

# Externe Validierung in §7 = prof_-Spalten + ausgeschlossene Features
validation_cols = prof_cols + excluded_cols

X = df[clustering_cols].values

print("Clustering auf:      ", clustering_cols)
print("Ausgeschlossen (§7): ", excluded_cols if excluded_cols else "- keine -")
print("Profil-Spalten:      ", prof_cols)
print("Shape gesamt:", df.shape, "| X:", X.shape)
print("Fehlende Werte gesamt:", int(df.isna().sum().sum()))
df.head()

In [ ]:
# Kurzer Skalen-Check: sollte Mittel ~0 und Std ~1 sein (aus M3)
df[clustering_cols].describe().T[["mean", "std", "min", "max"]].round(3)

## 3 · Kurze EDA: Verteilungen & Korrelationen

In [ ]:
ncols = 3
nrows = int(np.ceil(len(clustering_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.5 * nrows))
axes = np.atleast_1d(axes).ravel()
for ax, col in zip(axes, clustering_cols):
    sns.histplot(df[col], bins=40, kde=True, ax=ax, color="#4C72B0")
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("")
for ax in axes[len(clustering_cols):]:
    ax.set_visible(False)
fig.suptitle("Verteilungen der standardisierten Cluster-Features", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7, 5.5))
sns.heatmap(df[clustering_cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, cbar_kws={"shrink": .8})
plt.title("Korrelationsmatrix der Cluster-Features")
plt.tight_layout(); plt.show()

## 4 · Helferfunktionen
Alle wiederverwendeten Bausteine an einer Stelle:

- `evaluate` – Kennzahlen je Verfahren sammeln (Silhouette, Davies-Bouldin, Noise)
- `plot_clusters` – Cluster in einer 2D-Projektion (PCA / t-SNE / UMAP)
- `silhouette_grid` / `silhouette_plot` – Silhouette-Diagramme ("Messer-Plots") je k bzw. für eine fertige Lösung
- `sil_on_X`, `largest_share`, `knee_eps` – Metriken/Parametrisierung für die Density-Verfahren
  (`sil_on_X(labels, idx=...)`, falls auf einer Projektion/Teilmenge geclustert wurde)
- `scan_k` / `pick_best_k` – **empirische Wahl der Clusterzahl** je Verfahren (Silhouette maximal, Davies-Bouldin als Tie-Break)
- `clustering_kappa` – **Cohen's Kappa** zwischen zwei Clusterlösungen (Hungarian-Label-Alignment, Noise ausgeschlossen)

In [ ]:
from matplotlib.lines import Line2D

results = []   # Sammel-Liste für die finale Vergleichstabelle (§6)

# 2D-Projektion EINMAL berechnen -> alle PCA-Panels vergleichbar
X_pca = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)

# Feste Silhouette-Stichprobe: silhouette_samples ist O(n^2). Wir ziehen EINMAL
# denselben Index-Satz und werten alle Lösungen darauf aus -> die Silhouette-
# Diagramme sind über Verfahren und k hinweg vergleichbar.
rng = np.random.default_rng(RANDOM_STATE)
sil_idx = rng.choice(len(X), size=min(SIL_SAMPLE, len(X)), replace=False)
X_sil = X[sil_idx]


def evaluate(name, labels):
    """Kennzahlen robust berechnen (Noise -1 ausschließen, >= 2 Cluster nötig)
    und in die Vergleichstabelle (results) eintragen."""
    labels = np.asarray(labels)
    mask = labels != -1
    uniq = set(labels[mask])
    n_clusters = len(uniq)
    n_noise = int((labels == -1).sum())
    if n_clusters >= 2 and mask.sum() > n_clusters:
        sil = silhouette_score(X[mask], labels[mask],
                               sample_size=min(SIL_SAMPLE, int(mask.sum())),
                               random_state=RANDOM_STATE)
        dbi = davies_bouldin_score(X[mask], labels[mask])
    else:
        sil, dbi = np.nan, np.nan
    results.append({"Algorithmus": name, "n_Cluster": n_clusters,
                    "Noise": n_noise, "Silhouette": sil, "Davies_Bouldin": dbi})
    print(f"{name:16s} | Cluster: {n_clusters:2d} | Noise: {n_noise:5d} "
          f"| Silhouette: {sil:.3f} | DB: {dbi:.3f}")
    return labels


def plot_clusters(labels, title, emb=None, idx=None, ax=None,
                  xlabel="PC1", ylabel="PC2"):
    """Cluster in einer 2D-Projektion – feste Farbe pro Cluster, beschriftetes
    Zentrum und Legende (inkl. Größe), damit die Segmente klar erkennbar sind.

    emb : 2D-Koordinaten (default: X_pca). idx : optionale Punkt-Auswahl
    (z. B. t-SNE-Stichprobe) – dann werden labels[idx] geplottet.
    ax  : zum Einbetten in ein Grid; ohne ax wird eine eigene Figur erzeugt.
    """
    labels = np.asarray(labels)
    if emb is None:
        emb = X_pca
    if idx is not None:
        labels = labels[idx]
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(7.5, 6))

    # Cluster nach Größe: große zuerst zeichnen, kleine kommen oben drauf -> sichtbar
    clusters = sorted((c for c in np.unique(labels) if c != -1),
                      key=lambda c: (labels == c).sum(), reverse=True)
    cmap = plt.get_cmap("tab10" if len(clusters) <= 10 else "tab20")
    colors = {c: cmap(i % cmap.N) for i, c in enumerate(clusters)}

    # Noise dezent in den Hintergrund
    noise = labels == -1
    if noise.any():
        ax.scatter(emb[noise, 0], emb[noise, 1], c="lightgrey",
                   s=6, alpha=.30, linewidths=0, zorder=1)

    handles = []
    for i, c in enumerate(clusters):
        m = labels == c
        ax.scatter(emb[m, 0], emb[m, 1], color=colors[c],
                   s=14, alpha=.65, linewidths=0, zorder=2 + i)
        # Cluster-Zentrum hervorheben + beschriften (nur als Anker fürs Label)
        cx, cy = emb[m, 0].mean(), emb[m, 1].mean()
        ax.scatter(cx, cy, marker="o", s=280, color=colors[c],
                   edgecolor="black", linewidths=1.8, zorder=50)
        ax.text(cx, cy, str(c), color="black", fontsize=10, fontweight="bold",
                ha="center", va="center", zorder=51)
        handles.append(Line2D([0], [0], marker="o", linestyle="", markersize=9,
                              markerfacecolor=colors[c], markeredgecolor="black",
                              label=f"Cluster {c}  (n={int(m.sum()):,})"))
    if noise.any():
        handles.append(Line2D([0], [0], marker="o", linestyle="", markersize=9,
                              markerfacecolor="lightgrey", markeredgecolor="none",
                              label=f"Noise  (n={int(noise.sum()):,})"))

    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(handles=handles, loc="best", framealpha=.9, fontsize=8)
    if own_fig:
        plt.tight_layout(); plt.show()

In [ ]:
# ---------------- Silhouette-Diagramme ("Messer-Plots") ----------------
# Lesart:
#   - breite "Messer" klar rechts der 0 und über der roten Ø-Linie -> saubere Cluster
#   - Werte links der 0 (negativ) -> Punkte, die besser in einen Nachbarcluster passten
#   - sehr unterschiedlich dicke Messer -> stark ungleiche Clustergrößen

def _draw_silhouette(ax, s_val, s_lab, title):
    """Zeichnet EIN Silhouette-Diagramm aus Per-Punkt-Silhouetten + Labels."""
    s_val = np.asarray(s_val); s_lab = np.asarray(s_lab)
    sil_avg = s_val.mean()
    clusters = sorted(np.unique(s_lab))
    cmap = plt.get_cmap("tab10" if len(clusters) <= 10 else "tab20")
    y_lower = 10
    for i, c in enumerate(clusters):
        vals = np.sort(s_val[s_lab == c])       # innerhalb des Clusters sortieren
        y_upper = y_lower + vals.shape[0]
        color = cmap(i % cmap.N)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals,
                         facecolor=color, edgecolor=color, alpha=.8)
        ax.text(-0.05, y_lower + 0.5 * vals.shape[0], str(c), va="center", fontsize=8)
        y_lower = y_upper + 10                   # Lücke zwischen den Clustern
    ax.axvline(sil_avg, color="red", ls="--", lw=1.2)   # Ø-Silhouette
    ax.set_title(f"{title}  (Ø Sil={sil_avg:.3f})", fontsize=10)
    ax.set_xlabel("Silhouette-Koeffizient")
    ax.set_xlim(-0.2, 1.0)
    ax.set_yticks([])
    ax.set_ylabel("Punkte, nach Cluster gruppiert")


def silhouette_grid(label_of_k, ks, title, ncols=3):
    """Silhouette-Diagramm je k (wie beim K-Means-Scan) für ein beliebiges
    Verfahren mit steuerbarer Clusterzahl. label_of_k(k) -> Labels ALLER Punkte;
    ausgewertet wird auf der festen Stichprobe (sil_idx / X_sil)."""
    ks = list(ks)
    nrows = int(np.ceil(len(ks) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.2 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, k in zip(axes, ks):
        labels = np.asarray(label_of_k(k))
        s_lab = labels[sil_idx]
        s_val = silhouette_samples(X_sil, s_lab)
        _draw_silhouette(ax, s_val, s_lab, f"k={k}")
    for ax in axes[len(ks):]:
        ax.set_visible(False)
    fig.suptitle(f"{title}  (Stichprobe n={len(sil_idx):,})",
                 fontsize=15, fontweight="bold")
    plt.tight_layout(); plt.show()


def silhouette_plot(labels, title, ax=None):
    """Silhouette-Diagramm für EINE fertige Clusterlösung (z. B. DBSCAN/HDBSCAN).
    Noise (-1) wird ausgeschlossen - die Silhouette ist für Ausreißer nicht
    definiert; bewertet wird nur die Qualität der eigentlichen Cluster."""
    labels = np.asarray(labels)
    m = labels != -1
    if len(set(labels[m].tolist())) < 2:
        print(f"{title}: < 2 Cluster ohne Noise - Silhouette-Diagramm nicht anwendbar.")
        return
    idx_pool = np.flatnonzero(m)
    rng_loc = np.random.default_rng(RANDOM_STATE)
    take = rng_loc.choice(idx_pool, size=min(SIL_SAMPLE, len(idx_pool)), replace=False)
    s_lab = labels[take]
    if len(set(s_lab.tolist())) < 2:            # Stichprobe könnte Cluster verlieren
        take = idx_pool
        s_lab = labels[take]
    s_val = silhouette_samples(X[take], s_lab)
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(6.5, 4.5))
    _draw_silhouette(ax, s_val, s_lab, title)
    if own_fig:
        plt.tight_layout(); plt.show()

In [ ]:
# ---------------- Metriken & Parametrisierung für Density-Verfahren ----------------

def sil_on_X(labels, idx=None):
    """Silhouette IMMER im echten Feature-Raum X (fairer, vergleichbarer Maßstab),
    egal worauf geclustert wurde. Noise (-1) ausgeschlossen.

    idx : Zeilen von X, zu denen `labels` gehören (default: alle Punkte).
          Nötig, wenn auf einer Teilmenge oder einer 2D-Projektion geclustert
          wurde - dann müssen die Labels auf DIESELBEN Zeilen von X zeigen.
    """
    labels = np.asarray(labels)
    X_ref = X if idx is None else X[np.asarray(idx)]
    if len(labels) != len(X_ref):
        raise ValueError(
            f"sil_on_X: {len(labels)} Labels passen nicht zu {len(X_ref)} Zeilen in X. "
            "Ursache ist meist eine veraltete Projektion (X_tsne/X_umap/tsne_idx) aus "
            "einem früheren Lauf mit anderer Datenmenge -> §5.6/§5.7 neu ausführen "
            "oder Kernel neu starten."
        )
    m = labels != -1
    if len(set(labels[m].tolist())) < 2:
        return np.nan
    try:
        return silhouette_score(X_ref[m], labels[m],
                                sample_size=min(SIL_SAMPLE, int(m.sum())),
                                random_state=RANDOM_STATE)
    except ValueError:
        return np.nan


def largest_share(labels):
    """Anteil des größten Clusters an ALLEN Punkten in % (entlarvt Mega-Cluster)."""
    labels = np.asarray(labels); nn = labels[labels != -1]
    return round(100 * np.bincount(nn).max() / len(labels), 1) if len(nn) else np.nan


def scan_k(label_of_k, ks=None):
    """Silhouette & Davies-Bouldin je k für ein Verfahren mit steuerbarer Clusterzahl.
    label_of_k(k) -> Labels ALLER Punkte. Rückgabe: (metrics_df, labels_dict), damit
    die beste Lösung ohne erneutes Fitten weiterverwendet werden kann."""
    ks = list(K_RANGE if ks is None else ks)
    rows, labs = [], {}
    for k in ks:
        lab = np.asarray(label_of_k(k))
        labs[k] = lab
        rows.append({"k": k,
                     "Silhouette": silhouette_score(X, lab,
                                                    sample_size=min(SIL_SAMPLE, len(X)),
                                                    random_state=RANDOM_STATE),
                     "Davies_Bouldin": davies_bouldin_score(X, lab)})
    return pd.DataFrame(rows).set_index("k"), labs


def pick_best_k(metrics_df):
    """Empirische Wahl der Clusterzahl: höchste Silhouette,
    bei Gleichstand der niedrigste Davies-Bouldin."""
    ranked = metrics_df.sort_values(["Silhouette", "Davies_Bouldin"],
                                    ascending=[False, True])
    return int(ranked.index[0])


def knee_eps(data, ms):
    """eps am Knie der k-Distance-Kurve.
    Die Kurve ist konvex (flach -> steiler Anstieg), liegt also UNTER der Sehne.
    Das Knie = Punkt mit maximalem Abstand UNTER der Sehne -> argmax(line - kd)."""
    nn = NearestNeighbors(n_neighbors=ms).fit(data)
    kd = np.sort(nn.kneighbors(data)[0][:, -1])
    line = np.linspace(kd[0], kd[-1], len(kd))
    eps = kd[np.argmax(line - kd)]
    return eps, kd

In [ ]:
# ---------------- Cohen's Kappa zwischen Clusterlösungen ----------------
# Cluster-Nummern sind willkürlich (Cluster 0 von K-Means != Cluster 0 von Ward).
# Für Kappa müssen die Labelsätze daher zuerst aufeinander abgebildet werden:
# Hungarian-Matching maximiert die Übereinstimmung in der Kontingenztafel.
# Noise (-1) wird beidseitig ausgeschlossen (Kappa vergleicht nur zugeordnete Punkte).

def align_labels(ref, other):
    """Bildet die Cluster-Nummern von `other` per Hungarian-Matching auf die
    Nummern von `ref` ab; überzählige Cluster erhalten neue, eindeutige Nummern."""
    ref = np.asarray(ref); other = np.asarray(other)
    ru, ou = np.unique(ref), np.unique(other)
    C = np.zeros((len(ou), len(ru)), dtype=np.int64)
    for i, oc in enumerate(ou):
        mo = other == oc
        for j, rc in enumerate(ru):
            C[i, j] = int(np.sum(mo & (ref == rc)))
    rows, cols = linear_sum_assignment(-C)      # Übereinstimmung maximieren
    mapping = {int(ou[i]): int(ru[j]) for i, j in zip(rows, cols)}
    nxt = int(ru.max()) + 1
    out = np.empty(len(other), dtype=int)
    for i, l in enumerate(other):
        l = int(l)
        if l not in mapping:
            mapping[l] = nxt
            nxt += 1
        out[i] = mapping[l]
    return out


def clustering_kappa(a, b):
    """Cohen's Kappa zwischen zwei Clusterlösungen.
    Rückgabe: (kappa, Anteil gemeinsam zugeordneter Punkte)."""
    a = np.asarray(a); b = np.asarray(b)
    m = (a != -1) & (b != -1)
    if m.sum() == 0:
        return np.nan, 0.0
    b_aligned = align_labels(a[m], b[m])
    return cohen_kappa_score(a[m], b_aligned), float(m.mean())

## 5 · Algorithmen-Vergleich
Für jeden Algorithmus: fitten → Cluster im PCA-Raum plotten → Silhouette-Diagramm → Kennzahlen sammeln.
Das **finale Modell** je Verfahren wird mit der **empirisch besten Clusterzahl** gefittet: über `k = 2 … 8`
werden Silhouette (höher = besser) und Davies-Bouldin (niedriger = besser) berechnet; gewählt wird das k mit
der höchsten Silhouette (Davies-Bouldin als Tie-Break). Das finale Modell wandert per `evaluate()` in die
Vergleichstabelle (§6).
Density-Verfahren (DBSCAN/HDBSCAN) bestimmen die Clusterzahl selbst und markieren Ausreißer als **Noise** (Label −1) –
diese Ausreißer werden in §5.4/§5.5 explizit identifiziert und profiliert.


### 5.1 - K-Means

In [ ]:
ks = list(K_RANGE)
ncols = 3
nrows = int(np.ceil(len(ks) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.6 * nrows))
axes = axes.ravel()

for ax, k in zip(axes, ks):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    sil = silhouette_score(X, km.labels_,
                           sample_size=min(SIL_SAMPLE, len(X)),
                           random_state=RANDOM_STATE)
    plot_clusters(km.labels_, f"k={k}  (Sil={sil:.3f})", emb=X_pca, ax=ax)

for ax in axes[len(ks):]:
    ax.set_visible(False)

fig.suptitle(f"K-Means auf X (PCA-2D-Projektion), k = {ks[0]} … {ks[-1]}",
             fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Empirische k-Wahl für K-Means: Kennzahlen je k, dann bestes k (Silhouette
# maximal, Davies-Bouldin als Tie-Break). Die Labels je k werden gemerkt, damit
# das finale Modell nicht erneut gefittet werden muss.
km_metrics, km_labels = scan_k(lambda k: KMeans(n_clusters=k, n_init=10,
                                                random_state=RANDOM_STATE).fit_predict(X))
display(km_metrics.T.round(3))

best_k_km = pick_best_k(km_metrics)
print(f"Bestes k (K-Means): {best_k_km}")

# Silhouette-Diagramme je k (gleiche feste Stichprobe wie bei Ward/GMM)
silhouette_grid(lambda k: km_labels[k], K_RANGE, "Silhouette-Diagramme je k (K-Means)")

# Finales K-Means-Modell (k = best_k_km) -> Vergleichstabelle + Labels für §6/§7
labels_km = evaluate("K-Means", km_labels[best_k_km])
plot_clusters(labels_km, f"K-Means (k={best_k_km})")


### 5.2 · Hierarchical (Agglomerative, Ward)
Dendrogramm auf einer Zufalls-Stichprobe (Ward/Linkage ist $O(n^2)$ Speicher – für 21k unpraktisch, zur *Visualisierung* ist eine Stichprobe Standard). Das eigentliche Clustering schneiden wir per `fcluster` aus dem EINMAL auf allen Punkten berechneten Ward-Baum.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(X), size=min(2000, len(X)), replace=False)
Z = linkage(X[idx], method="ward")

plt.figure(figsize=(11, 4))
dendrogram(Z, truncate_mode="level", p=5, no_labels=True, color_threshold=None)
plt.title(f"Dendrogramm (Ward, Stichprobe n={len(idx):,})")
plt.xlabel("Beispiele"); plt.ylabel("Ward-Distanz")
plt.tight_layout(); plt.show()

In [ ]:
# Welches k bevorzugt Ward? (analog zur empirischen k-Wahl bei K-Means in §5.1)
# Ward-Linkage EINMAL auf allen Punkten berechnen, dann je k schneiden (effizient).
Z_full = linkage(X, method="ward")          # O(n^2) Speicher, auf 21k noch machbar
ward_k = range(2, 9)
w_sil, w_dbi = [], []
for k in ward_k:
    lab = fcluster(Z_full, t=k, criterion="maxclust")
    w_sil.append(silhouette_score(X, lab, sample_size=min(SIL_SAMPLE, len(X)),
                                  random_state=RANDOM_STATE))
    w_dbi.append(davies_bouldin_score(X, lab))

# Ward-"Elbow": Ward hat kein SSE/Inertia wie K-Means. Das hierarchische Pendant sind
# die MERGE-DISTANZEN (Höhen im Dendrogramm) – die Distanz, bei der die Daten in k
# Cluster zerfallen. Ein Knick markiert (wie der SSE-Elbow) eine natürliche Clusterzahl.
heights_desc = np.sort(Z_full[:, 2])[::-1]
ward_merge = [heights_desc[k - 2] for k in ward_k]   # Höhe, um k Cluster zu bilden

ward_metrics = pd.DataFrame({"k": list(ward_k), "Merge_Distanz": ward_merge,
                             "Silhouette": w_sil, "Davies_Bouldin": w_dbi}).set_index("k")

w_best_sil = ward_metrics["Silhouette"].idxmax()
w_best_dbi = ward_metrics["Davies_Bouldin"].idxmin()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(list(ward_k), ward_merge, "o-", color="#4C72B0"); ax[0].set_title("Merge-Distanz – Knick suchen")
ax[1].plot(list(ward_k), w_sil, "o-", color="#55A868");     ax[1].set_title("Silhouette – höher = besser")
ax[2].plot(list(ward_k), w_dbi, "o-", color="#C44E52");     ax[2].set_title("Davies-Bouldin – niedriger = besser")

ax[1].axvline(w_best_sil, ls="--", c="grey", alpha=.7)
ax[1].annotate(f"max @ k={w_best_sil}", (w_best_sil, ward_metrics.loc[w_best_sil, "Silhouette"]),
               textcoords="offset points", xytext=(6, -10), fontsize=8)
ax[2].axvline(w_best_dbi, ls="--", c="grey", alpha=.7)
ax[2].annotate(f"min @ k={w_best_dbi}", (w_best_dbi, ward_metrics.loc[w_best_dbi, "Davies_Bouldin"]),
               textcoords="offset points", xytext=(6, 6), fontsize=8)

for a in ax:
    a.set_xlabel("k")
plt.tight_layout(); plt.show()

best_k_ward = pick_best_k(ward_metrics[["Silhouette", "Davies_Bouldin"]])
print(f"Ward-Optimum: Silhouette bei k={w_best_sil}, Davies-Bouldin bei k={w_best_dbi}")
print(f"Bestes k (Ward): {best_k_ward}")
ward_metrics.T.round(3)

In [ ]:
ks = list(range(2, 10))
ncols = 3
nrows = int(np.ceil(len(ks) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.6 * nrows))
axes = axes.ravel()

for ax, k in zip(axes, ks):
    lab = fcluster(Z_full, t=k, criterion="maxclust")          # k Cluster aus dem Baum schneiden
    sil = silhouette_score(X, lab, sample_size=min(SIL_SAMPLE, len(X)),
                           random_state=RANDOM_STATE)
    plot_clusters(lab, f"Ward k={k}  (Sil={sil:.3f})", emb=X_pca, ax=ax)

for ax in axes[len(ks):]:
    ax.set_visible(False)

fig.suptitle(f"Ward (hierarchisch) auf X (PCA-2D-Projektion), k = {ks[0]} … {ks[-1]}",
             fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Silhouette-Diagramme je k für Ward (analog zu §5.1 bei K-Means)
silhouette_grid(lambda k: fcluster(Z_full, t=k, criterion="maxclust"),
                range(2, 9), "Silhouette-Diagramme je k (Ward)")

In [ ]:
# Finales Ward-Modell (k = best_k_ward, empirisch aus dem Scan oben)
# -> Vergleichstabelle + Labels für §6/§7
labels_agg = evaluate("Hierarchical", fcluster(Z_full, t=best_k_ward, criterion="maxclust"))
plot_clusters(labels_agg, f"Ward (k={best_k_ward})")


### 5.3 · Gaussian Mixture Model (GMM)
Probabilistisches Pendant zu K-Means: weiche Zuordnung über Normalverteilungs-Komponenten,
harte Labels via `predict`. Auch hier sind Silhouette-Diagramme je k anwendbar.

In [ ]:
# Silhouette-Diagramme je k für GMM (analog zu §5.1 bei K-Means)
silhouette_grid(lambda k: GaussianMixture(n_components=k, n_init=3,
                                          random_state=RANDOM_STATE).fit_predict(X),
                range(2, 9), "Silhouette-Diagramme je k (GMM)")

In [ ]:
# Empirische k-Wahl für GMM (analog zu §5.1/§5.2), dann finales Modell mit
# mehr Initialisierungen (n_init=5) beim gewählten k.
gmm_metrics, gmm_labels = scan_k(lambda k: GaussianMixture(n_components=k, n_init=3,
                                                           random_state=RANDOM_STATE).fit_predict(X))
display(gmm_metrics.T.round(3))

best_k_gmm = pick_best_k(gmm_metrics)
print(f"Bestes k (GMM): {best_k_gmm}")

# Finales GMM (k = best_k_gmm) -> Vergleichstabelle + Labels für §6/§7
labels_gmm = evaluate("GMM", GaussianMixture(n_components=best_k_gmm, n_init=5,
                                             random_state=RANDOM_STATE).fit_predict(X))
plot_clusters(labels_gmm, f"GMM (k={best_k_gmm})")


### 5.4 · DBSCAN
`eps` bestimmen wir über die **k-Distance-Kurve** (Distanz zum `min_samples`-ten Nachbarn, sortiert). Der Knick markiert eine sinnvolle Nachbarschafts-Größe. Faustregel `min_samples ≈ 2 · Anzahl Features`.

In [ ]:
min_samples = 2 * X.shape[1]                 # Faustregel 2*D  (D = Anzahl Cluster-Features)
eps, kdist = knee_eps(X, min_samples)
print(f"D = {X.shape[1]},  min_samples = {min_samples},  gewähltes eps = {eps:.3f}")

plt.figure(figsize=(7, 4))
plt.plot(np.arange(len(kdist)), kdist, color="#4C72B0")
plt.axhline(eps, ls="--", c="red", label=f"eps = {eps:.2f}")
plt.xlabel("sortierte Punkte"); plt.ylabel(f"Distanz zum {min_samples}. Nachbarn")
plt.title("k-Distance-Plot (DBSCAN eps-Bestimmung)"); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
db = DBSCAN(eps=eps, min_samples=min_samples).fit(X)
labels_db = evaluate("DBSCAN", db.labels_)
plot_clusters(labels_db, f"DBSCAN (eps={eps:.2f}, min_samples={min_samples})")

In [ ]:
# Silhouette-Diagramm der DBSCAN-Lösung (Noise ausgeschlossen, s. Helfer in §4).
# Achtung Lesart: Da DBSCAN dichte-basierte, beliebig geformte Cluster findet,
# ist die (zentroid-orientierte) Silhouette hier nur ein grober Anhaltspunkt.
silhouette_plot(labels_db, "DBSCAN (ohne Noise)")

In [ ]:
# Hängt das Ergebnis an min_samples? Faustregel 2*D vs. Untergrenze D+1,
# je einmal mit festem eps (isoliert min_samples) und mit neu abgeleitetem eps.
ms_rule = 2 * X.shape[1]
ms_min  = X.shape[1] + 1
eps_ref, _ = knee_eps(X, ms_rule)

def dbscan_eval(ms, eps_val):
    lab = DBSCAN(eps=eps_val, min_samples=ms).fit(X).labels_
    nc = len(set(lab)) - (1 if -1 in lab else 0)
    noise = int((lab == -1).sum())
    return nc, noise, round(noise / len(X) * 100, 1)

rows = []
for ms in (ms_rule, ms_min):
    for src, e in [("eps fest (Regel)", eps_ref), ("eps neu abgeleitet", knee_eps(X, ms)[0])]:
        nc, no, pc = dbscan_eval(ms, e)
        rows.append({"min_samples": ms, "eps": round(float(e), 3), "eps_Quelle": src,
                     "Cluster": nc, "Noise": no, "Noise_%": pc})
sens = pd.DataFrame(rows)
display(sens)

nl = chr(10)
labels_bar = [f"ms={r.min_samples}{nl}{r.eps_Quelle}" for r in sens.itertuples()]
plt.figure(figsize=(8, 3.6))
bars = plt.bar(labels_bar, sens["Noise_%"],
               color=["#C44E52", "#C44E52", "#4C72B0", "#4C72B0"])
plt.ylabel("Noise-Anteil (%)"); plt.ylim(0, 100)
plt.title("DBSCAN: Sensitivität gegenüber min_samples / eps")
for b, v in zip(bars, sens["Noise_%"]):
    plt.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.1f}%", ha="center", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
def plot_dbscan_panel(ms, eps_val, ax):
    lab = DBSCAN(eps=eps_val, min_samples=ms).fit(X).labels_
    noise = lab == -1
    nc = len(set(lab)) - (1 if -1 in lab else 0)
    ax.scatter(X_pca[noise, 0], X_pca[noise, 1], c="lightgrey", s=5, alpha=.30, linewidths=0)
    ax.scatter(X_pca[~noise, 0], X_pca[~noise, 1], c=lab[~noise], cmap="tab20",
               s=9, alpha=.75, linewidths=0)
    ax.set_title(f"min_samples={ms}:  {nc} Cluster,  {noise.mean()*100:.1f}% Noise")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
plot_dbscan_panel(ms_rule, eps_ref, axes[0])   # Faustregel 2*D
plot_dbscan_panel(ms_min,  eps_ref, axes[1])   # Untergrenze D+1, gleiches eps
fig.suptitle("DBSCAN: Effekt von min_samples bei gleichem eps  (grau = Noise)",
             fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# =============================================================================
# "k = 2..10 bei DBSCAN?" - DBSCAN hat KEIN k als Parameter! Die Clusterzahl ist
# ein ERGEBNIS aus (eps, min_samples). Wir drehen an eps, bis DBSCAN möglichst
# genau die Zielzahl liefert; min_samples bleibt fest = 2*n_features.
# KEIN evaluate() -> Haupt-Vergleichstabelle (results) bleibt unberührt.
# =============================================================================
_ms_db = 2 * X.shape[1]

# eps-Raster aus der k-Distance-Verteilung: klein (Mikro-Cluster) bis groß (k=1)
_nn_db = NearestNeighbors(n_neighbors=_ms_db).fit(X)
_kd_db = np.sort(_nn_db.kneighbors(X)[0][:, -1])
eps_scan = np.linspace(np.percentile(_kd_db, 30), np.percentile(_kd_db, 99.5), 45)

# ---- db_runs AUFBAUEN: je eps einmal DBSCAN, k/Noise/Silhouette merken ----
db_runs = []
for e in eps_scan:
    lab = DBSCAN(eps=float(e), min_samples=_ms_db).fit(X).labels_
    db_runs.append({"eps": float(e),
                    "k": len(set(lab.tolist()) - {-1}),
                    "noise": float((lab == -1).mean()),
                    "sil": sil_on_X(lab)})
db_runs = pd.DataFrame(db_runs)

# ---- Für jede Ziel-Clusterzahl die nächstbeste eps-Konfiguration wählen ----
rows = []
for target in range(2, 11):
    cand = db_runs.assign(dist=(db_runs["k"] - target).abs())
    best = cand.sort_values(["dist", "noise"]).iloc[0]
    lab_b = DBSCAN(eps=float(best["eps"]), min_samples=_ms_db).fit(X).labels_
    rows.append({"Ziel_k": target,
                 "erreichtes_k": int(best["k"]),
                 "getroffen": "ja" if int(best["k"]) == target else "nein",
                 "eps": round(float(best["eps"]), 3),
                 "min_samples": _ms_db,
                 "Noise_%": round(best["noise"] * 100, 1),
                 "größtes_Cl_%": largest_share(lab_b),
                 "Silhouette_auf_X": round(best["sil"], 3) if pd.notna(best["sil"]) else np.nan})
dbscan_k_table = pd.DataFrame(rows).set_index("Ziel_k")

# ---- Visualisierung ----
tk = dbscan_k_table.index.to_numpy()
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].plot(tk, tk, "--", color="grey", lw=1, label="ideal (erreicht = Ziel)")
ax[0].plot(tk, dbscan_k_table["erreichtes_k"], "o-", color="#4C72B0", label="tatsächlich erreicht")
ax[0].set(title="Erreichte vs. gewünschte Clusterzahl", xlabel="Ziel-k", ylabel="erreichtes k")
ax[0].legend(fontsize=8)
ax[1].plot(tk, dbscan_k_table["größtes_Cl_%"], "o-", color="#C44E52")
ax[1].axhline(90, color="grey", ls=":", lw=1)
ax[1].set(title="Größtes Cluster (% aller Punkte)", xlabel="Ziel-k", ylabel="%")
ax[2].axhline(0, color="black", lw=.7)
ax[2].plot(tk, dbscan_k_table["Silhouette_auf_X"], "o-", color="#55A868")
ax[2].set(title="Silhouette (auf X) der DBSCAN-Lösung", xlabel="Ziel-k", ylabel="Silhouette")
fig.suptitle("DBSCAN 'auf k=2..10 gebracht': eps getunt, min_samples fix = %d" % _ms_db,
             fontsize=14, y=1.03)
plt.tight_layout(); plt.show()

print("DBSCAN auf Ziel-Clusterzahl gebracht (min_samples fix = %d):" % _ms_db)
print(dbscan_k_table)
print("\nLesart: DBSCAN kennt kein k - die Zielzahl wird nur durch eps-Drehen angenähert.")
print("'größtes_Cl_%' entlarvt, ob eine Lösung nur ein Mega-Cluster + Reste ist.")

#### 5.4.1 · Ausreißer-Identifikation (DBSCAN)
DBSCAN liefert die Ausreißer direkt mit: alle Punkte mit Label **−1 (Noise)** liegen in keiner
ausreichend dichten Nachbarschaft. Wir identifizieren diese Punkte explizit, geben ihnen einen
**Ausreißer-Score** (Distanz zum `min_samples`-ten Nachbarn – dieselbe Größe, aus der `eps` abgeleitet
wurde: je größer, desto isolierter) und profilieren sie gegen die Inlier.

In [ ]:
# ---- Ausreißer-Flag + Score in df ablegen (für §5.5.2 und §7 wiederverwendbar) ----
outlier_db = labels_db == -1
nn_all = NearestNeighbors(n_neighbors=min_samples).fit(X)
kdist_all = nn_all.kneighbors(X)[0][:, -1]          # Distanz zum min_samples-ten Nachbarn

df["outlier_dbscan"] = outlier_db
df["dbscan_kdist"]   = kdist_all

print(f"DBSCAN-Ausreißer: {outlier_db.sum():,} von {len(X):,} Punkten "
      f"({outlier_db.mean()*100:.1f} %)")

# ---- Profil: Wodurch fallen die Ausreißer auf? (Ø standardisierte Features) ----
out_prof = pd.DataFrame({
    "Ausreißer": df.loc[outlier_db, clustering_cols].mean(),
    "Inlier":    df.loc[~outlier_db, clustering_cols].mean(),
})
out_prof["Differenz"] = out_prof["Ausreißer"] - out_prof["Inlier"]
display(out_prof.round(2).sort_values("Differenz", key=np.abs, ascending=False))

# ---- Top 10 extremste Ausreißer (größte Nachbar-Distanz), inkl. Validierungs-Spalten ----
top_out = (df.loc[outlier_db, clustering_cols + validation_cols + ["dbscan_kdist"]]
             .nlargest(10, "dbscan_kdist"))
print("\nTop 10 extremste DBSCAN-Ausreißer (nach k-Distanz):")
display(top_out.round(2))

In [ ]:
# ---- Visualisierung: Ausreißer im PCA-Raum + Score-Verteilung ----
fig, ax = plt.subplots(1, 2, figsize=(13.5, 5))

ax[0].scatter(X_pca[~outlier_db, 0], X_pca[~outlier_db, 1], c="lightgrey",
              s=6, alpha=.35, linewidths=0, label=f"Inlier (n={int((~outlier_db).sum()):,})")
ax[0].scatter(X_pca[outlier_db, 0], X_pca[outlier_db, 1], c="#C44E52",
              s=14, alpha=.8, linewidths=0, label=f"Ausreißer (n={int(outlier_db.sum()):,})")
ax[0].set(title="DBSCAN-Ausreißer im PCA-Raum", xlabel="PC1", ylabel="PC2")
ax[0].legend(fontsize=8)

ax[1].hist(kdist_all[~outlier_db], bins=60, color="lightgrey", label="Inlier", density=True)
ax[1].hist(kdist_all[outlier_db], bins=60, color="#C44E52", alpha=.7, label="Ausreißer", density=True)
ax[1].axvline(eps, ls="--", c="black", lw=1, label=f"eps = {eps:.2f}")
ax[1].set(title="Ausreißer-Score: Distanz zum %d. Nachbarn" % min_samples,
          xlabel="k-Distanz", ylabel="Dichte")
ax[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

### 5.5 · HDBSCAN
Erweitert DBSCAN über verschiedene Dichte-Skalen und braucht kein festes `eps`. Wir steuern nur die minimale Clustergröße.

In [ ]:
hdb = HDBSCAN(min_cluster_size=250, min_samples=15, copy=True).fit(X)
labels_hdb = evaluate("HDBSCAN", hdb.labels_)
plot_clusters(labels_hdb, "HDBSCAN")

In [ ]:
# Silhouette-Diagramm der HDBSCAN-Lösung (Noise ausgeschlossen, gleiche Lesart wie §5.4)
silhouette_plot(labels_hdb, "HDBSCAN (ohne Noise)")

#### 5.5.1 · Ausreißer-Identifikation (HDBSCAN)
Auch HDBSCAN markiert Ausreißer als **Noise (−1)**. Zusätzlich liefert `probabilities_` die
**Zuordnungs-Sicherheit** je Punkt (0 = Noise/unsicher, 1 = Kern des Clusters). Damit lassen sich
neben harten Ausreißern auch **Randpunkte** (schwach zugeordnet) erkennen.

In [ ]:
outlier_hdb = labels_hdb == -1
df["outlier_hdbscan"] = outlier_hdb
df["hdbscan_prob"]    = hdb.probabilities_

WEAK_P = 0.10     # Schwelle für "schwach zugeordnete" Randpunkte
weak = (~outlier_hdb) & (hdb.probabilities_ < WEAK_P)

print(f"HDBSCAN-Ausreißer (Noise):        {outlier_hdb.sum():,} "
      f"({outlier_hdb.mean()*100:.1f} %)")
print(f"Schwach zugeordnet (p < {WEAK_P}):   {weak.sum():,} "
      f"({weak.mean()*100:.1f} %)")

# ---- Profil: Ausreißer vs. Inlier (Ø standardisierte Features) ----
out_prof_h = pd.DataFrame({
    "Ausreißer": df.loc[outlier_hdb, clustering_cols].mean(),
    "Inlier":    df.loc[~outlier_hdb, clustering_cols].mean(),
})
out_prof_h["Differenz"] = out_prof_h["Ausreißer"] - out_prof_h["Inlier"]
display(out_prof_h.round(2).sort_values("Differenz", key=np.abs, ascending=False))

# ---- Visualisierung: Ausreißer/Randpunkte im PCA-Raum + Verteilung der Zuordnungs-Sicherheit ----
fig, ax = plt.subplots(1, 2, figsize=(13.5, 5))

core = ~outlier_hdb & ~weak
ax[0].scatter(X_pca[core, 0], X_pca[core, 1], c="lightgrey", s=6, alpha=.35,
              linewidths=0, label=f"Kern (n={int(core.sum()):,})")
ax[0].scatter(X_pca[weak, 0], X_pca[weak, 1], c="#DD8452", s=12, alpha=.8,
              linewidths=0, label=f"schwach zugeordnet (n={int(weak.sum()):,})")
ax[0].scatter(X_pca[outlier_hdb, 0], X_pca[outlier_hdb, 1], c="#C44E52", s=14, alpha=.8,
              linewidths=0, label=f"Ausreißer/Noise (n={int(outlier_hdb.sum()):,})")
ax[0].set(title="HDBSCAN: Ausreißer & Randpunkte im PCA-Raum", xlabel="PC1", ylabel="PC2")
ax[0].legend(fontsize=8)

ax[1].hist(hdb.probabilities_, bins=40, color="#4C72B0")
ax[1].axvline(WEAK_P, ls="--", c="#DD8452", label=f"Schwelle p = {WEAK_P}")
ax[1].set(title="Zuordnungs-Sicherheit (probabilities_)",
          xlabel="p  (0 = Noise/unsicher, 1 = Cluster-Kern)", ylabel="Anzahl Punkte")
ax[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

#### 5.5.2 · Konsens-Ausreißer: DBSCAN vs. HDBSCAN
Punkte, die **beide** Density-Verfahren als Noise markieren, sind besonders verlässliche Ausreißer-Kandidaten
(z. B. für Datenbereinigung oder eine Sonderbehandlung im Geschäftsprozess).

In [ ]:
both   = outlier_db & outlier_hdb
only_d = outlier_db & ~outlier_hdb
only_h = ~outlier_db & outlier_hdb
union  = outlier_db | outlier_hdb
jaccard = both.sum() / union.sum() if union.sum() else np.nan

print(pd.crosstab(df["outlier_dbscan"], df["outlier_hdbscan"],
                  rownames=["DBSCAN-Ausreißer"], colnames=["HDBSCAN-Ausreißer"]))
print(f"\nKonsens-Ausreißer (beide Verfahren): {both.sum():,} "
      f"({both.mean()*100:.2f} %)  |  Jaccard-Überlappung: {jaccard:.2f}")

df["outlier_konsens"] = both

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_pca[~union, 0], X_pca[~union, 1], c="lightgrey", s=6, alpha=.3,
           linewidths=0, label=f"Inlier (n={int((~union).sum()):,})")
ax.scatter(X_pca[only_d, 0], X_pca[only_d, 1], c="#4C72B0", s=12, alpha=.8,
           linewidths=0, label=f"nur DBSCAN (n={int(only_d.sum()):,})")
ax.scatter(X_pca[only_h, 0], X_pca[only_h, 1], c="#DD8452", s=12, alpha=.8,
           linewidths=0, label=f"nur HDBSCAN (n={int(only_h.sum()):,})")
ax.scatter(X_pca[both, 0], X_pca[both, 1], c="#C44E52", s=18, alpha=.9,
           linewidths=0, label=f"Konsens (n={int(both.sum()):,})")
ax.set(title="Ausreißer-Vergleich DBSCAN vs. HDBSCAN (PCA-Raum)", xlabel="PC1", ylabel="PC2")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# Wodurch fallen die Konsens-Ausreißer auf? (inkl. Validierungs-Spalten)
if both.any():
    cons_prof = pd.DataFrame({
        "Konsens-Ausreißer": df.loc[both, clustering_cols + validation_cols].mean(),
        "Inlier":            df.loc[~union, clustering_cols + validation_cols].mean(),
    })
    cons_prof["Differenz"] = cons_prof["Konsens-Ausreißer"] - cons_prof["Inlier"]
    display(cons_prof.round(2).sort_values("Differenz", key=np.abs, ascending=False))

### 5.6 · t-SNE-Projektion (nichtlinear, nur zur Visualisierung)
PC1/PC2 fangen nur einen Teil der Varianz ein, weshalb die Cluster im PCA-Bild stark überlappen. **t-SNE** ist eine nichtlineare Projektion, die lokale Nachbarschaften erhält und die Segmente optisch meist deutlich klarer trennt.

Wie die PCA dient t-SNE **nur der Visualisierung** – geclustert wird weiterhin auf allen Cluster-Features. Wichtig: t-SNE-Achsen und -Abstände sind **nicht** quantitativ interpretierbar (keine feste Bedeutung, kein „größer = teurer"), sie zeigen nur, *ob* sich Gruppen sauber separieren lassen.

In [ ]:
from sklearn.manifold import TSNE

# t-SNE EINMAL berechnen (~30-40 s) und im Kernel zwischenspeichern.
# WICHTIG: Der Cache darf nur wiederverwendet werden, wenn er zur AKTUELLEN
# Datenmenge passt. Ohne diese Prüfung überlebt eine Projektion aus einem
# früheren Lauf (z. B. mit einer Stichprobe) den Wechsel auf den vollen
# Datensatz und passt dann nicht mehr zu X -> IndexError in §5.8.
_tsne_ok = ("X_tsne" in globals() and "tsne_idx" in globals()
            and len(X_tsne) == len(X) and len(tsne_idx) == len(X))
if not _tsne_ok:
    tsne_idx = np.arange(len(X))
    X_tsne = TSNE(n_components=2, perplexity=30, init="pca",
                  learning_rate="auto", random_state=RANDOM_STATE).fit_transform(X[tsne_idx])
    print("t-SNE neu berechnet für", len(tsne_idx), "Punkte")
else:
    print("t-SNE aus dem Cache übernommen:", len(tsne_idx), "Punkte")

# Alle finalen Verfahren (§5.1-§5.5) im t-SNE-Raum
runs = [("K-Means", labels_km), ("Hierarchical", labels_agg), ("GMM", labels_gmm),
        ("DBSCAN", labels_db), ("HDBSCAN", labels_hdb)]

ncols = 3
nrows = int(np.ceil(len(runs) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5.5 * nrows))
axes = axes.ravel()
for ax, (name, lab) in zip(axes, runs):
    plot_clusters(lab, name, emb=X_tsne, idx=tsne_idx, ax=ax,
                  xlabel="t-SNE 1", ylabel="t-SNE 2")
for ax in axes[len(runs):]:
    ax.set_visible(False)
fig.suptitle(f"Cluster im t-SNE-Raum (alle n={len(tsne_idx):,} Objekte)", fontsize=15, y=1.0)
plt.tight_layout(); plt.show()

In [ ]:
def emb_kgrid(emb, idx, label_of_k, ks, title, xlabel="Dim 1", ylabel="Dim 2", ncols=3):
    """Zeigt je k eine Clusterlösung in einer beliebigen 2D-Projektion emb."""
    nrows = int(np.ceil(len(ks) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.6 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, k in zip(axes, ks):
        lab = label_of_k(k)
        sil = silhouette_score(X, lab, sample_size=min(SIL_SAMPLE, len(X)),
                               random_state=RANDOM_STATE)
        plot_clusters(lab, f"k={k}  (Sil={sil:.3f})", emb=emb, idx=idx, ax=ax,
                      xlabel=xlabel, ylabel=ylabel)
    for ax in axes[len(ks):]:
        ax.set_visible(False)
    fig.suptitle(title, fontsize=16, fontweight="bold")
    plt.tight_layout(); plt.show()

In [ ]:
emb_kgrid(X_tsne, tsne_idx,
          lambda k: KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X),
          list(range(2, 10)), "K-Means im t-SNE-Raum, k = 2 … 9",
          xlabel="t-SNE 1", ylabel="t-SNE 2")

In [ ]:
emb_kgrid(X_tsne, tsne_idx,
          lambda k: fcluster(Z_full, t=k, criterion="maxclust"),
          list(range(2, 10)), "Ward (hierarchisch) im t-SNE-Raum, k = 2 … 9",
          xlabel="t-SNE 1", ylabel="t-SNE 2")

### 5.7 · UMAP-Projektion (Vergleich zu t-SNE)
**UMAP** ist eine zweite nichtlineare Projektion. Gegenüber t-SNE erhält es neben der lokalen auch die **globale** Struktur meist besser (Abstände zwischen Clustern sind etwas aussagekräftiger) und ist schneller. Wir rechnen es auf **denselben Punkten** wie t-SNE, damit der Vergleich fair ist.

Auch UMAP dient **nur der Visualisierung** – geclustert wird weiter auf allen Cluster-Features, und die Achsenwerte sind nicht quantitativ interpretierbar. Zeigen beide Projektionen (t-SNE *und* UMAP) dieselben Gruppen, ist das ein starkes Indiz, dass die Segmente **real** sind und kein Projektionsartefakt.

In [ ]:
import umap
print("NumPy:", np.__version__, "| umap:", umap.__version__)

# UMAP (random_state -> reproduzierbar, aber single-threaded => etwas langsamer, ~40-60 s)
# Gleiche Cache-Prüfung wie bei t-SNE: nur wiederverwenden, wenn die Länge zu X passt.
assert len(tsne_idx) == len(X), "tsne_idx passt nicht zu X - §5.6 neu ausführen."
_umap_ok = "X_umap" in globals() and len(X_umap) == len(tsne_idx)
if not _umap_ok:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        X_umap = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2,
                           random_state=RANDOM_STATE).fit_transform(X[tsne_idx])
    print("UMAP neu berechnet für", len(tsne_idx), "Punkte")
else:
    print("UMAP aus dem Cache übernommen:", len(tsne_idx), "Punkte")

# a) t-SNE vs. UMAP am selben Clustering (finales K-Means-Modell)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
plot_clusters(labels_km, "K-Means · t-SNE", emb=X_tsne, idx=tsne_idx, ax=axes[0],
              xlabel="t-SNE 1", ylabel="t-SNE 2")
plot_clusters(labels_km, "K-Means · UMAP", emb=X_umap, idx=tsne_idx, ax=axes[1],
              xlabel="UMAP 1", ylabel="UMAP 2")
fig.suptitle(f"Projektions-Vergleich: t-SNE vs. UMAP (gleiche Punkte, K-Means k={best_k_km})",
             fontsize=15, y=1.02)
plt.tight_layout(); plt.show()

# b) alle finalen Verfahren im UMAP-Raum
runs = [("K-Means", labels_km), ("Hierarchical", labels_agg), ("GMM", labels_gmm),
        ("DBSCAN", labels_db), ("HDBSCAN", labels_hdb)]
ncols = 3
nrows = int(np.ceil(len(runs) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5.5 * nrows))
axes = axes.ravel()
for ax, (name, lab) in zip(axes, runs):
    plot_clusters(lab, name, emb=X_umap, idx=tsne_idx, ax=ax, xlabel="UMAP 1", ylabel="UMAP 2")
for ax in axes[len(runs):]:
    ax.set_visible(False)
fig.suptitle(f"Cluster im UMAP-Raum (alle n={len(tsne_idx):,} Objekte)", fontsize=15, y=1.0)
plt.tight_layout(); plt.show()

In [ ]:
emb_kgrid(X_umap, tsne_idx,
          lambda k: KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X),
          list(range(2, 10)), "K-Means im UMAP-Raum, k = 2 … 9",
          xlabel="UMAP 1", ylabel="UMAP 2")

In [ ]:
emb_kgrid(X_umap, tsne_idx,
          lambda k: fcluster(Z_full, t=k, criterion="maxclust"),
          list(range(2, 10)), "Ward im UMAP-Raum, k = 2 … 9",
          xlabel="UMAP 1", ylabel="UMAP 2")

### 5.8 · HDBSCAN: beste Konfiguration (auf X vs. auf dem Embedding)
HDBSCAN kennt kein `k` – die Clusterzahl ist ein **Ergebnis** aus `min_cluster_size`/`min_samples`.
Statt eine feste Zielzahl vorzugeben, suchen wir je Variante die `min_cluster_size` mit der **besten
Silhouette (gemessen auf X)** unter der Nebenbedingung, dass höchstens `MAX_NOISE_FRAC` der Punkte
als Noise markiert werden (sonst wäre es keine vollständige Segmentierung).


In [ ]:
def hdbscan_best(data, idx=None, min_samples=10,
                 sizes=(250, 400, 600, 900, 1400, 1600, 2000, 3000)):
    """Sucht die min_cluster_size mit der besten Silhouette (IMMER auf X gemessen,
    damit alle Varianten vergleichbar sind), unter der Nebenbedingung
    Noise <= MAX_NOISE_FRAC und >= 2 Cluster. Tie-Break: weniger Noise.
    HDBSCAN wählt selbst, WIE VIELE Cluster entstehen; über die minimale
    Clustergröße steuern wir nur die Auflösung.

    data : Raum, in dem geclustert wird (X oder eine 2D-Projektion).
    idx  : Zeilen von X, die `data` entspricht (bei Projektionen: tsne_idx),
           damit die Silhouette auf den RICHTIGEN Zeilen von X gemessen wird.
    """
    if idx is not None and len(data) != len(idx):
        raise ValueError(f"data ({len(data)}) und idx ({len(idx)}) passen nicht zusammen.")
    runs = []
    for mcs in sizes:
        lab = HDBSCAN(min_cluster_size=int(mcs), min_samples=min_samples,
                      copy=True).fit(data).labels_
        runs.append({"mcs": int(mcs), "k": len(set(lab) - {-1}),
                     "noise": float((lab == -1).mean()),
                     "sil": sil_on_X(lab, idx=idx), "labels": lab})
    ok = [r for r in runs
          if r["k"] >= 2 and r["noise"] <= MAX_NOISE_FRAC and pd.notna(r["sil"])]
    pool = ok if ok else [r for r in runs if pd.notna(r["sil"])] or runs
    pool = sorted(pool, key=lambda r: (-(r["sil"] if pd.notna(r["sil"]) else -np.inf),
                                       r["noise"]))
    return pool[0], runs

# Varianten: A auf X; B auf den 2D-Projektionen.
# Die Projektionen beziehen sich auf die Zeilen tsne_idx von X -> idx mitgeben,
# damit die Silhouette auf denselben Zeilen von X gemessen wird.
variants = [("A_X", f"A · HDBSCAN auf X ({X.shape[1]} Feat.)", X, None),
            ("B_tsne", "B · HDBSCAN auf t-SNE (2D)", X_tsne, tsne_idx),
            ("B_umap", "B · HDBSCAN auf UMAP (2D)", X_umap, tsne_idx)]

rows, labels_by = [], {}
for tag, name, data, idx in variants:
    best, _ = hdbscan_best(data, idx=idx)
    labels_by[tag] = best["labels"]
    rows.append({"Variante": name, "min_cluster_size": best["mcs"],
                 "n_Cluster": best["k"],
                 "Noise_%": round(best["noise"] * 100, 1),
                 "größtes_Cl_%": largest_share(best["labels"]),
                 "Silhouette_auf_X": round(sil_on_X(best["labels"], idx=idx), 3)})
hdb_summary = pd.DataFrame(rows).set_index("Variante")

# Labels für spätere Plots bereitstellen
labels_hdb_X    = labels_by.get("A_X")
labels_hdb_tsne = labels_by.get("B_tsne")
labels_hdb_umap = labels_by.get("B_umap")

print(f"Auswahlkriterium: max. Silhouette auf X  (Noise <= {MAX_NOISE_FRAC:.0%}, >= 2 Cluster)")
display(hdb_summary)
print(f"\nSilhouette bei ALLEN Varianten im echten {X.shape[1]}-Feature-Raum X gemessen,")
print("damit A (auf X) und B (auf 2D-Projektion) vergleichbar sind.")
print("'größtes_Cl_%' entlarvt, ob eine Lösung nur ein Mega-Cluster + Reste ist.")


In [ ]:
# HDBSCAN "auf k=2..9 gebracht": min_cluster_size variieren (k ist ein ERGEBNIS).
# Sweep EINMAL über ein mcs-Raster, dann je Ziel-k die nächste Lösung wählen.
mcs_grid = [100, 150, 200, 300, 400, 600, 800, 1100, 1500, 2000, 2800, 4000, 6000]
hdb_min_samples = 10

sweep = []
for mcs in mcs_grid:
    lab = HDBSCAN(min_cluster_size=int(mcs), min_samples=hdb_min_samples, copy=True).fit(X).labels_
    sweep.append({"mcs": int(mcs), "k": len(set(lab) - {-1}),
                  "noise": float((lab == -1).mean()), "labels": lab})

rows = []
for target in range(2, 10):
    best = sorted(sweep, key=lambda r: (abs(r["k"] - target), r["noise"]))[0]
    rows.append({"Ziel_k": target, "erreichtes_k": best["k"],
                 "getroffen": "ja" if best["k"] == target else "nein",
                 "min_cluster_size": best["mcs"],
                 "Noise_%": round(best["noise"] * 100, 1),
                 "größtes_Cl_%": largest_share(best["labels"]),
                 "Silhouette_auf_X": sil_on_X(best["labels"])})
hdb_k_table = pd.DataFrame(rows).set_index("Ziel_k")
display(hdb_k_table.round(3))

tk = hdb_k_table.index.to_numpy()
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].plot(tk, tk, "--", color="grey", lw=1, label="ideal (erreicht = Ziel)")
ax[0].plot(tk, hdb_k_table["erreichtes_k"], "o-", color="#4C72B0", label="erreicht")
ax[0].set(title="Erreichte vs. gewünschte Clusterzahl", xlabel="Ziel-k", ylabel="erreichtes k"); ax[0].legend(fontsize=8)
ax[1].plot(tk, hdb_k_table["größtes_Cl_%"], "o-", color="#C44E52"); ax[1].axhline(90, color="grey", ls=":", lw=1)
ax[1].set(title="Größtes Cluster (% aller Punkte)", xlabel="Ziel-k", ylabel="%")
ax[2].axhline(0, color="black", lw=.7)
ax[2].plot(tk, hdb_k_table["Silhouette_auf_X"], "o-", color="#55A868")
ax[2].set(title="Silhouette (auf X)", xlabel="Ziel-k", ylabel="Silhouette")
fig.suptitle("HDBSCAN 'auf k=2..9 gebracht' (min_cluster_size getunt, min_samples=%d)" % hdb_min_samples,
             fontsize=14, y=1.03)
plt.tight_layout(); plt.show()

## 6 · Bewertung & Vergleich der Algorithmen
Silhouette (höher = besser) und Davies-Bouldin (niedriger = besser). Noise = als Ausreißer markierte Punkte.
Alle finalen Modelle wurden bereits in §5 gefittet und per `evaluate()` gesammelt.

In [ ]:
# Defensive Absicherung: Falls §5 nur teilweise gelaufen ist, fehlende finale
# Modelle hier nachfitten - inkl. empirischer k-Wahl (normalerweise ist diese
# Zelle ein No-Op).
_done = {r["Algorithmus"] for r in results}
if "K-Means" not in _done:
    km_metrics, km_labels = scan_k(lambda k: KMeans(n_clusters=k, n_init=10,
                                                    random_state=RANDOM_STATE).fit_predict(X))
    best_k_km = pick_best_k(km_metrics)
    labels_km = evaluate("K-Means", km_labels[best_k_km])
if "Hierarchical" not in _done:
    if "Z_full" not in globals():
        Z_full = linkage(X, method="ward")
    _ward_metrics, _ward_labels = scan_k(lambda k: fcluster(Z_full, t=k, criterion="maxclust"))
    best_k_ward = pick_best_k(_ward_metrics)
    labels_agg = evaluate("Hierarchical", _ward_labels[best_k_ward])
if "GMM" not in _done:
    gmm_metrics, gmm_labels = scan_k(lambda k: GaussianMixture(n_components=k, n_init=3,
                                                               random_state=RANDOM_STATE).fit_predict(X))
    best_k_gmm = pick_best_k(gmm_metrics)
    labels_gmm = evaluate("GMM", GaussianMixture(n_components=best_k_gmm, n_init=5,
                                                 random_state=RANDOM_STATE).fit_predict(X))

comparison = pd.DataFrame(results).set_index("Algorithmus")
comparison = comparison.sort_values("Silhouette", ascending=False)
comparison.round(3)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
comparison["Silhouette"].plot.bar(ax=ax[0], color="#55A868")
ax[0].set_title("Silhouette (höher = besser)"); ax[0].tick_params(axis="x", rotation=30)
comparison["Davies_Bouldin"].plot.bar(ax=ax[1], color="#C44E52")
ax[1].set_title("Davies-Bouldin (niedriger = besser)"); ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

### 6.1 · Silhouette-Diagramme aller finalen Modelle
Direkter visueller Vergleich der finalen Lösungen. Bei DBSCAN/HDBSCAN wird Noise ausgeschlossen
(Silhouette ist für Ausreißer nicht definiert) – die Diagramme bewerten dort nur die gefundenen Cluster.

In [ ]:
final_models = [("K-Means", labels_km), ("Ward", labels_agg), ("GMM", labels_gmm),
                ("DBSCAN (ohne Noise)", labels_db), ("HDBSCAN (ohne Noise)", labels_hdb)]

ncols = 3
nrows = int(np.ceil(len(final_models) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5.4 * ncols, 4.4 * nrows))
axes = axes.ravel()
for ax, (name, lab) in zip(axes, final_models):
    silhouette_plot(lab, name, ax=ax)
for ax in axes[len(final_models):]:
    ax.set_visible(False)
fig.suptitle(f"Silhouette-Diagramme der finalen Modelle (Stichprobe max. n={SIL_SAMPLE:,})",
             fontsize=15, fontweight="bold")
plt.tight_layout(); plt.show()

### 6.2 · Übereinstimmung der Verfahren: Cohen's Kappa
**Cohen's Kappa** misst die über den Zufall hinausgehende Übereinstimmung zweier Labelings
(1 = perfekt, 0 = Zufallsniveau). Da Cluster-Nummern willkürlich sind, richten wir die Labels
zuerst per Hungarian-Matching aus (§4, `clustering_kappa`); Noise-Punkte werden paarweise
ausgeschlossen. Zum Vergleich zeigen wir den **Adjusted Rand Index (ARI)** – das Standardmaß
für Clustering-Übereinstimmung, das ohne Label-Alignment auskommt.

Interpretation: Hohe Kappa-Werte zwischen unterschiedlichen Verfahren = die Segmentstruktur
ist **robust** und kein Artefakt eines einzelnen Algorithmus.

In [ ]:
model_labels = {"K-Means": labels_km, "Hierarchical": labels_agg, "GMM": labels_gmm,
                "DBSCAN": labels_db, "HDBSCAN": labels_hdb}
names = list(model_labels)

kappa_mat = pd.DataFrame(np.eye(len(names)), index=names, columns=names)
ari_mat   = pd.DataFrame(np.eye(len(names)), index=names, columns=names)
for i, a in enumerate(names):
    for b in names[i + 1:]:
        kap, shared = clustering_kappa(model_labels[a], model_labels[b])
        la, lb = np.asarray(model_labels[a]), np.asarray(model_labels[b])
        m = (la != -1) & (lb != -1)
        ari = adjusted_rand_score(la[m], lb[m])
        kappa_mat.loc[a, b] = kappa_mat.loc[b, a] = kap
        ari_mat.loc[a, b]   = ari_mat.loc[b, a]   = ari

fig, ax = plt.subplots(1, 2, figsize=(14, 5.2))
sns.heatmap(kappa_mat, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1,
            square=True, cbar_kws={"shrink": .8}, ax=ax[0])
ax[0].set_title("Cohen's Kappa (Labels ausgerichtet, Noise paarweise ausgeschlossen)")
sns.heatmap(ari_mat, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1,
            square=True, cbar_kws={"shrink": .8}, ax=ax[1])
ax[1].set_title("Adjusted Rand Index (zum Vergleich)")
plt.tight_layout(); plt.show()

print("Paarweise Übereinstimmung der finalen Clusterlösungen:")
display(kappa_mat.round(3))

### 6.3 · Stabilität des finalen K-Means (Cohen's Kappa über Seeds)
Zweite sinnvolle Kappa-Anwendung: Wie stabil ist die K-Means-Lösung gegenüber der
Zufalls-Initialisierung? Wir fitten das finale Modell mit verschiedenen Seeds und messen
Kappa gegen die Referenzlösung. Werte nahe 1 → die Segmentierung ist reproduzierbar.

In [ ]:
seeds = range(RANDOM_STATE + 1, RANDOM_STATE + 11)
stab = []
for s in seeds:
    lab_s = KMeans(n_clusters=best_k_km, n_init=10, random_state=s).fit_predict(X)
    kap, _ = clustering_kappa(labels_km, lab_s)
    stab.append({"seed": s, "Kappa_vs_Referenz": kap})
stab = pd.DataFrame(stab).set_index("seed")

plt.figure(figsize=(8, 3.4))
plt.bar(stab.index.astype(str), stab["Kappa_vs_Referenz"], color="#4C72B0")
plt.axhline(1.0, color="grey", ls="--", lw=1)
plt.ylim(0, 1.05)
plt.ylabel("Cohen's Kappa"); plt.xlabel("random_state")
plt.title(f"Stabilität K-Means (k={best_k_km}): Kappa gegen Referenz (seed={RANDOM_STATE})")
plt.tight_layout(); plt.show()

print(f"Kappa über {len(stab)} Seeds:  Ø {stab['Kappa_vs_Referenz'].mean():.3f}  "
      f"| min {stab['Kappa_vs_Referenz'].min():.3f}")

### 6.4 · Auswahl & Visualisierung der zwei besten Clusterings

In [ ]:
# --- Auswahl der ZWEI BESTEN Clusterings ---
comp = pd.DataFrame(results).set_index("Algorithmus")
comp["Noise_%"] = (comp["Noise"] / len(X) * 100).round(1)

# Für eine Käufer-SEGMENTIERUNG muss (fast) jeder Punkt einem Segment gehören.
# Verfahren, die mehr als MAX_NOISE_FRAC der Punkte als Noise abstempeln, wären keine
# vollständige Segmentierung und werden ausgeschlossen.
valid = comp[comp["Noise"] / len(X) <= MAX_NOISE_FRAC]
ranked = valid.sort_values(["Silhouette", "Davies_Bouldin"], ascending=[False, True])
best_names = list(ranked.head(2).index)

print(f"Vollständige Segmentierungen (Noise <= {MAX_NOISE_FRAC:.0%}), Rangfolge nach Silhouette:")
print(ranked[["n_Cluster", "Noise_%", "Silhouette", "Davies_Bouldin"]].round(3))
print("\nAusgeschlossen (zu viel Noise -> keine echte Segmentierung):",
      list(comp.index.difference(valid.index)))
print(">> Die zwei besten Clusterings:", best_names)

# --- Genau diese zwei in t-SNE UND UMAP visualisieren ---
label_map = {"K-Means": labels_km, "Hierarchical": labels_agg, "GMM": labels_gmm,
             "DBSCAN": labels_db, "HDBSCAN": labels_hdb}

fig, axes = plt.subplots(len(best_names), 2, figsize=(13, 6 * len(best_names)))
axes = np.atleast_2d(axes)
for r, name in enumerate(best_names):
    plot_clusters(label_map[name], f"{name} · t-SNE", emb=X_tsne, idx=tsne_idx,
                  ax=axes[r, 0], xlabel="t-SNE 1", ylabel="t-SNE 2")
    plot_clusters(label_map[name], f"{name} · UMAP", emb=X_umap, idx=tsne_idx,
                  ax=axes[r, 1], xlabel="UMAP 1", ylabel="UMAP 2")
fig.suptitle(f"Die zwei besten Clusterings in t-SNE & UMAP: {best_names[0]} & {best_names[1]}",
             fontsize=15, y=1.0)
plt.tight_layout(); plt.show()

## 7 · Cluster-Profile & externe Validierung
Profiliert wird das finale **K-Means**-Modell (`labels_km`). Zwei Ebenen:

- **Profil A:** die standardisierten Cluster-Features (Basis der Segmentierung)
- **Profil B (externe Validierung):** die `prof_`-Spalten **und die in §1 ausgeschlossenen Features** –
  beides wurde im Clustering **nicht** verwendet. Trennen sich diese Größen trotzdem klar zwischen
  den Clustern, sind die Segmente auch inhaltlich echt, nicht nur geometrisch.

In [ ]:
prof = df.copy()
prof["cluster"] = labels_km

# Profil A: die standardisierten Cluster-Features (Basis der Segmentierung)
profile = prof.groupby("cluster")[clustering_cols].mean()
sizes = prof["cluster"].value_counts().sort_index()
profile.insert(0, "n", sizes)
display(profile.round(2))

# Profil B: externe Validierung über prof_-Spalten + ausgeschlossene Features
profile_ext = prof.groupby("cluster")[validation_cols].mean()
profile_ext.insert(0, "n", sizes)
print("Externe Validierung (im Clustering NICHT verwendet):", validation_cols)
display(profile_ext.round(3))

In [ ]:
# Validierung der AUSGESCHLOSSENEN Features (falls in §1 gesetzt):
# Boxplots je Cluster - klare Niveau-Unterschiede = die Segmente tragen auch
# auf Features, die der Algorithmus nie gesehen hat.
if excluded_cols:
    ncols = min(3, len(excluded_cols))
    nrows = int(np.ceil(len(excluded_cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, col in zip(axes, excluded_cols):
        sns.boxplot(data=prof, x="cluster", y=col, hue="cluster",
                    palette="tab10", legend=False, ax=ax)
        ax.set_title(col)
    for ax in axes[len(excluded_cols):]:
        ax.set_visible(False)
    fig.suptitle("Externe Validierung: ausgeschlossene Features je Cluster",
                 fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()
else:
    print("Keine Features in §1 ausgeschlossen (EXCLUDED_FEATURES ist leer) - "
          "externe Validierung läuft nur über die prof_-Spalten.")

In [ ]:
# Interpretation je Cluster (aus den Profil-Mittelwerten oben abzuleiten).
# Da k jetzt EMPIRISCH bestimmt wird (best_k_km), sind die Namen zunächst
# neutrale Platzhalter. Nach inhaltlicher Interpretation der Profile können
# hier sprechende Segment-Namen vergeben werden, z. B.:
#   segment_names.update({0: "Luxus", 1: "Urban kompakt"})
segment_names = {c: f"Segment {c}" for c in profile.index}

hm = profile.drop(columns="n").T          # Features = Zeilen (y), Cluster = Spalten (x)
xlabels = [f"Cluster {c}\n{segment_names.get(c, '?')}\n(n={int(profile.loc[c, 'n']):,})"
           for c in hm.columns]

plt.figure(figsize=(10, 4.2))
ax = sns.heatmap(hm, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
                 cbar_kws={"shrink": .8})
ax.set_xticklabels(xlabels, rotation=0, fontsize=8)
plt.title(f"Cluster-Profile & Segment-Zuordnung (Ø standardisierte Features je Cluster, k={best_k_km})")
plt.xlabel("Cluster / Segment"); plt.ylabel("Feature")
plt.tight_layout(); plt.show()


### 7.1 · Klarere Segment-Darstellung: Snake-Plot & Radar-Charts
Da der t-SNE/UMAP-Scatter durch die Überlappung unscharf ist, zeigen wir die Trennschärfe über die **Cluster-Mittelwerte** – das macht die Segmente eindeutig sichtbar.

In [ ]:
# Snake-Plot: Profil-Linie je Cluster über alle Features (Klassiker der Segmentierung).
# Zeigt die Trennschärfe der Segmente viel klarer als der überlappende t-SNE/UMAP-Scatter,
# weil hier die CLUSTER-MITTELWERTE verglichen werden, nicht die Einzelpunkte.
prof_only = profile.drop(columns="n")
cmap = plt.get_cmap("tab10")

plt.figure(figsize=(11, 4.8))
for c in prof_only.index:
    plt.plot(prof_only.columns, prof_only.loc[c], "o-", linewidth=2, color=cmap(c % 10),
             label=f"Cluster {c} - {segment_names.get(c, '?')} (n={int(profile.loc[c, 'n']):,})")
plt.axhline(0, color="grey", lw=1, ls="--")           # 0 = Gesamtdurchschnitt
plt.xticks(rotation=30, ha="right")
plt.ylabel("Ø standardisierter Wert  (+ über / - unter Durchschnitt)")
plt.title(f"Snake-Plot: Segment-Profile je Cluster (K-Means, k={best_k_km})")
plt.legend(fontsize=8, loc="best")
plt.tight_layout(); plt.show()

In [ ]:
# Radar-Charts: ein Netz je Segment (Persona-Darstellung).
# Gleiche radiale Skala für alle -> die Formen sind direkt vergleichbar.
feats = list(profile.drop(columns="n").columns)
N = len(feats)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]                      # Kreis schließen

vals_all = profile.drop(columns="n").values
gmin, gmax = float(vals_all.min()) - 0.3, float(vals_all.max()) + 0.3
cmap = plt.get_cmap("tab10")

ncols = 2
nrows = int(np.ceil(len(profile.index) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 5.5 * nrows), subplot_kw=dict(polar=True))
axes = np.atleast_1d(axes).ravel()
for ax, c in zip(axes, profile.index):
    vals = profile.drop(columns="n").loc[c].tolist()
    vals += vals[:1]
    ax.plot(angles, vals, "o-", linewidth=2, color=cmap(c % 10))
    ax.fill(angles, vals, color=cmap(c % 10), alpha=0.25)
    ax.plot(angles, [0] * len(angles), color="grey", lw=1, ls="--")   # 0 = Durchschnitt
    ax.set_ylim(gmin, gmax)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(feats, fontsize=7)
    ax.set_yticklabels([])
    ax.set_title(f"Cluster {c} - {segment_names.get(c, '?')}  (n={int(profile.loc[c, 'n']):,})",
                 fontsize=11, pad=22)
for ax in axes[len(profile.index):]:
    ax.set_visible(False)
fig.suptitle(f"Segment-Steckbriefe als Radar-Charts (Ø standardisierte Features, k={best_k_km})",
             fontsize=14, y=1.0)
plt.tight_layout(); plt.show()

## 7b · Rohdaten-Rückführung für die Kartenvisualisierung

`df` enthält nur standardisierte Features – für eine echte Geo-Karte werden aber reale `lat`/`long`-Koordinaten sowie Zusatzinfos (`zipcode`, `bedrooms`, `bathrooms`, `price`) benötigt.

Diese Zelle lädt die unbearbeiteten Rohdaten (`kc_house_data.csv`) und ordnet jeder Zeile in `df` die passende Rohdaten-Zeile zu — per Nächster-Nachbar-Suche auf den (rück-standardisierten) Koordinaten, eindeutig gemacht über einen exakten Abgleich von `price`/`grade`/`waterfront` (den einzigen unveränderten Werten aus `df`). Außerdem werden die Cluster-Label-Spalten (`Cluster_KMeans` etc.), die §8 erwartet, hier an `df` angehängt — sie wurden bisher nur als lose Arrays (`labels_km`, ...) gehalten.

**Voraussetzung:** `kc_house_data.csv` (der unbearbeitete King-County-Datensatz) liegt im selben Verzeichnis wie dieses Notebook.

In [ ]:
from scipy.spatial import cKDTree

# ---- 1) Rohdaten laden & auf denselben Skalenraum wie df['lat']/df['long'] bringen ----
raw = pd.read_csv("kc_house_data.csv")

lat_mean, lat_std   = raw["lat"].mean(),  raw["lat"].std(ddof=0)
long_mean, long_std = raw["long"].mean(), raw["long"].std(ddof=0)
raw["lat_z"]  = (raw["lat"]  - lat_mean)  / lat_std
raw["long_z"] = (raw["long"] - long_mean) / long_std

# ---- 2) Für jede df-Zeile den passenden Rohdaten-Datensatz finden ----
# Kandidaten per Radius-Suche in der Koordinaten-Nachbarschaft, dann eindeutig gemacht
# über exakten Abgleich von price/grade/waterfront (unverändert aus df verfügbar als prof_-Spalten).
tree = cKDTree(raw[["lat_z", "long_z"]].values)

matched_idx, n_fallback = [], 0
for _, row in df.iterrows():
    cand_idx = tree.query_ball_point([row["lat"], row["long"]], r=0.01)
    if not cand_idx:
        matched_idx.append(None)
        n_fallback += 1
        continue
    cands = raw.iloc[cand_idx]
    exact = cands[(cands["price"] == row["prof_price"]) &
                  (cands["grade"] == row["prof_grade"]) &
                  (cands["waterfront"] == row["prof_waterfront"])]
    pool = exact if len(exact) else cands
    if not len(exact):
        n_fallback += 1
    d = (pool["lat_z"] - row["lat"]) ** 2 + (pool["long_z"] - row["long"]) ** 2
    matched_idx.append(pool.loc[d.idxmin()].name)

print(f"Zeilen ohne exakten price/grade/waterfront-Treffer (Fallback auf nächste Koordinate): {n_fallback} / {len(df)}")

# ---- 3) df_map zusammenbauen (reale Koordinaten + Zusatzinfos, gleiche Reihenfolge wie df) ----
df_map = raw.loc[matched_idx, ["lat", "long", "price", "zipcode", "bedrooms", "bathrooms"]].reset_index(drop=True)
df_map.columns = ["lat", "long", "price", "zipcode", "bedrooms_orig", "bathrooms_orig"]
df_map.index = df.index

# ---- 4) Cluster-Label-Spalten an df anhängen (von §8 erwartet: Cluster_<Modellname ohne Bindestrich>) ----
df["Cluster_KMeans"]       = labels_km
df["Cluster_Hierarchical"] = labels_agg
df["Cluster_GMM"]          = labels_gmm
df["Cluster_DBSCAN"]       = labels_db
df["Cluster_HDBSCAN"]      = labels_hdb

df_map.head()

## 8. Geographical Visualization of Clusters

In [ ]:
# 2. Interactive Map using Folium
import sys

# Auto-install folium if missing
try:
    import folium
except ImportError:
    print("Installing folium...")
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "folium"])
    import folium


def display_all_clusters_map(model_name):
    """
    Plots all properties on a Seattle map, colored by their cluster label.
    """
    model_col = f"Cluster_{model_name.replace('-', '')}"
    if model_col not in df.columns:
        print(f"Error: Column {model_col} not found. Please run clustering first.")
        return

    print(f"Plotting all clusters for {model_name} ({len(df)} properties total)...")

    # Plot all properties (no sampling)
    plot_data = df.copy()

    # Align coordinates using our preloaded df_map
    plot_data['lat'] = df_map.loc[plot_data.index, 'lat']
    plot_data['long'] = df_map.loc[plot_data.index, 'long']
    plot_data['price'] = df_map.loc[plot_data.index, 'price']
    plot_data['zipcode'] = df_map.loc[plot_data.index, 'zipcode']
    plot_data['bedrooms_orig'] = df_map.loc[plot_data.index, 'bedrooms_orig']
    plot_data['bathrooms_orig'] = df_map.loc[plot_data.index, 'bathrooms_orig']

    # Center map on Seattle, enable all standard zoom functions
    seattle_lat, seattle_long = 47.6062, -122.3321
    m = folium.Map(location=[seattle_lat, seattle_long], zoom_start=11, tiles="cartodbpositron")

    # Palette for up to 10 clusters (distinct hex colors)
    palette = ['#3B82F6', '#F59E0B', '#10B981', '#EF4444', '#4F46E5', '#8B5CF6', '#EC4899', '#14B8A6', '#06B6D4',
               '#84CC16']

    for idx, row in plot_data.iterrows():
        price_val = row['price']
        price_str = f"${price_val:,.0f}"
        cluster_val = int(row[model_col])

        # Determine color and label
        if cluster_val == -1:
            color = "#94A3B8"  # Slate grey for noise
            cluster_name = "Noise (Outlier)"
        else:
            color = palette[cluster_val % len(palette)]
            cluster_name = f"Cluster {cluster_val}"

        html_popup = f'''
        <div style="font-family: Arial, sans-serif; font-size: 12px; line-height: 1.4;">
            <h4 style="margin: 0 0 5px 0; color: {color};">{cluster_name}</h4>
            <b>Price:</b> {price_str}<br>
            <b>Zipcode:</b> {int(row['zipcode'])}<br>
            <b>Bedrooms:</b> {int(row['bedrooms_orig'])}<br>
            <b>Bathrooms:</b> {row['bathrooms_orig']}<br>
            <b>Luxury Score:</b> {row['luxury_score']:.2f}<br>
        </div>
        '''

        # Add CircleMarker directly to map without MarkerCluster
        folium.CircleMarker(
            location=[row['lat'], row['long']],
            radius=2.0,
            weight=0,
            popup=folium.Popup(html_popup, max_width=200),
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.6
        ).add_to(m)

    # Add a simple HTML legend
    legend_html = f'''
    <div style="
        position: fixed;
        bottom: 50px; left: 50px; width: 150px; height: auto;
        background-color: white; border:2px solid grey; z-index:9999;
        font-family: Arial; font-size:12px; padding: 10px;
        border-radius: 6px; box-shadow: 2px 2px 5px rgba(0,0,0,0.2);
    ">
    <h4 style="margin: 0 0 8px 0; font-size: 13px; color: #374151;">{model_name} Legend</h4>
    '''

    unique_clusters = sorted(plot_data[model_col].unique())
    for c in unique_clusters:
        c = int(c)
        if c == -1:
            color = "#94A3B8"
            name = "Noise"
        else:
            color = palette[c % len(palette)]
            name = f"Cluster {c}"
        legend_html += f'<div style="margin-bottom: 4px;"><span style="display:inline-block; width:12px; height:12px; background-color:{color}; border-radius:50%; margin-right:8px; vertical-align:middle;"></span>{name}</div>'

    legend_html += '</div>'
    m.get_root().html.add_child(folium.Element(legend_html))

    return m


# Choose preferred model to visualize (available: 'KMeans', 'GMM', 'DBSCAN', 'Agglo', 'HDBSCAN')
preferred_model = 'KMeans'

m = display_all_clusters_map(preferred_model)
m